# Glucose-Aware Food Recommender: Data Preparation
Cleans the NHANES 2017-2018 files and builds the transactions dataset.

## Part 1: Setup, load, clean
### Step 1: Imports and settings

In [1]:
import os, sys, re, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

In [2]:
# SETTINGS YOU CAN CHANGE

# Where the XPT files live
DATA_DIR = os.environ.get("NHANES_DIR", "/content/drive/MyDrive/NHANES")
# Where the finished CSV is saved
OUT_DIR  = os.environ.get("OUT_DIR", DATA_DIR)
OUT_FILE = "glucose_food_training_data_v3.csv"
NUT_FILE = "food_group_nutrition.csv"
ITEMS_FILE = "food_group_items.csv"
ITEM_BASKETS_FILE = "food_group_item_baskets.csv"
ITEM_MIN_PEOPLE = 5
ITEM_TOP_N = 15

# Who is in the dataset
MIN_AGE = 18
DAYS_TO_USE = (1, 2)
EXCLUDE_DIAGNOSED = True

# Glucose bands
BAND_CUTS   = [0, 100, 126, np.inf]
BAND_LABELS = ["Normal (<100)", "Prediabetic range (100-125)", "Diabetic range (>=126)"]

# Food grouping
# NOISE_GROUPS: removed from the dataset completely (in almost every basket, says nothing useful)
NOISE_GROUPS = {
    940, 941, 942, 943,   # Tap water, Bottled water, Flavored water, Baby water
    911, 912,             # Sugar, Sugar substitutes
    921,                  # Coffee
    122,                  # Coffee creamer
    110, 117,             # Human milk, Infant formula
    744, 755, 899,        # Ketchup and salsa, Mustard pickles and olives, Sandwich toppings
}

# ALWAYS_EXCLUDE: stays in the dataset, but never recommended (sweetened drinks)
ALWAYS_EXCLUDE = {953,    # Sports and energy drinks
                  924,    # Soft drinks
                  925,    # Fruit drinks
                  929,    # Drink mixes
                  923}    # Tea (mostly pre-sweetened)

# ALCOHOL_PREFIX: codes starting with 93 are never recommended
# 931 Beer, 932 Liqueur, 933 Cocktails, 934 Wine, 935 Liquor
ALCOHOL_PREFIX = 93

# FATS_PREFIXES: codes starting with 81, 82, 83 are never recommended (add-ons, not foods)
# 811 Butter and margarine, 812 Ghee and animal fat, 813 Tartar sauce and spreads,
# 821 Cooking oils, 831 Mayonnaise and dressings, 832 Light dressings, 833 Fat free dressings
FATS_PREFIXES  = {81, 82, 83}

# PROCESSED_EXCLUDE: fast food, fried food and salty snacks, never recommended
PROCESSED_EXCLUDE = {581,   # Pizza and pasta dishes
                     241,   # Chicken and chicken nuggets
                     275,   # Burgers and sandwiches
                     263,   # Shrimp (mostly fried)
                     714,   # French fries and hash browns
                     712,   # Potato chips
                     544,   # Corn chips and pretzels
                     754,   # Onion rings and vegetable dishes
                     521}   # Biscuits

# HIGHER_BAND_EXCLUDE: not recommended for the higher glucose bands (fast sugar)
HIGHER_BAND_EXCLUDE = {612,   # Citrus juice
                       641,   # Fruit juice
                       115}   # Chocolate milk

# Nutrition filter, per 100 g of the food group
NUTRI_LIMITS = {
    BAND_LABELS[0]: dict(sugar=15, netcarb=60),
    BAND_LABELS[1]: dict(sugar=10, netcarb=40),
    BAND_LABELS[2]: dict(sugar=8,  netcarb=30),
}
FIBER_EXCEPTION = dict(min_fiber=1.5, max_sugar=12)
SATFAT_MAX = 10

GROUP_NAME_OVERRIDES = {
    111: "Milk", 112: "Evaporated or condensed milk", 113: "Almond or soy milk",
    114: "Yogurt", 115: "Chocolate milk", 117: "Infant formula",
    118: "Cocoa or milk powder mix", 121: "Cream", 122: "Coffee creamer",
    123: "Sour cream and dips", 131: "Ice cream", 132: "Pudding",
    133: "Baby food custard", 134: "White sauce or gravy", 140: "Cheese unspecified",
    141: "Natural cheese", 142: "Cottage cheese", 143: "Cream cheese",
    144: "American cheese", 145: "Imitation cheese", 146: "Cheese dip and grilled cheese",
    147: "Cheese soup", 200: "Meat unspecified", 210: "Beef unspecified cut",
    211: "Beef steak", 213: "Beef ribs and oxtail", 214: "Roast beef",
    215: "Ground beef", 216: "Beef jerky and beef bacon", 217: "Baby food beef",
    220: "Pork unspecified cut", 221: "Pork chop", 222: "Pork steak or tenderloin",
    223: "Ham", 224: "Pork roast", 225: "Canadian bacon",
    226: "Bacon", 227: "Pork ribs and rinds", 228: "Baby food ham",
    230: "Lamb", 231: "Lamb chop and goat", 232: "Veal",
    233: "Venison and bison", 241: "Chicken and chicken nuggets", 242: "Turkey",
    243: "Duck", 244: "Game hen and quail", 247: "Baby food poultry",
    251: "Organ meats", 252: "Deli meats and sausages", 261: "Fish",
    262: "Squid and octopus", 263: "Shrimp", 271: "Meat and poultry in sauce",
    272: "Meat loaf and meat with rice", 273: "Stews and pot pies", 274: "Chicken salad and meat with vegetables",
    275: "Burgers and sandwiches", 276: "Baby food meat dinners", 281: "Frozen meat dinners",
    283: "Meat and vegetable soups", 285: "Gravy and savory sauces", 311: "Eggs boiled or fried",
    312: "Duck or quail eggs", 321: "Scrambled eggs and omelets", 322: "Egg breakfast sandwiches",
    323: "Egg soup", 324: "Egg white omelet", 330: "Egg substitute",
    334: "Egg substitute dishes", 411: "Beans", 412: "Baked beans refried beans and hummus",
    413: "Lentils and chickpeas", 414: "Soy sauce and tofu", 416: "Bean and lentil soup",
    418: "Meatless burgers and bacon bits", 421: "Nuts", 422: "Peanut butter",
    423: "Peanut butter sandwich", 424: "Coconut water and milk", 425: "Trail mix",
    431: "Seeds", 510: "Bread unspecified", 511: "White bread and rolls",
    513: "Wheat bread", 514: "Rye bread", 515: "Oat bread",
    516: "Multigrain bread", 518: "Gluten free bread", 521: "Biscuits",
    522: "Tortillas and cornbread", 523: "Muffins", 524: "Sweet quick breads",
    531: "Cakes and cupcakes", 532: "Cookies and brownies", 533: "Pies",
    534: "Pastries and fritters", 535: "Doughnuts and breakfast tarts", 536: "Coffee cake",
    537: "Cereal and granola bars", 538: "Baby cookies", 540: "Crackers unspecified",
    541: "Graham crackers", 542: "Low sodium crackers", 543: "Crackers",
    544: "Corn chips and pretzels", 551: "Pancakes", 552: "Waffles",
    553: "French toast", 554: "Crepes", 555: "Chinese pancake",
    556: "Dumplings", 557: "Rice cakes and dosa", 558: "Funnel cake",
    561: "Pasta and noodles", 562: "Rice and oatmeal", 570: "Oat cereal",
    571: "Low sugar breakfast cereal", 572: "Medium sugar breakfast cereal", 573: "High sugar breakfast cereal",
    574: "Other breakfast cereal", 576: "Raw oats and bran", 578: "Baby cereal",
    581: "Pizza and pasta dishes", 582: "Jelly sandwich and wraps", 583: "Frozen meals",
    584: "Noodle soups", 585: "Baby food pasta", 590: "Grain based meat substitute",
    611: "Citrus fruit", 612: "Citrus juice", 621: "Dried fruit",
    631: "Fresh fruit", 632: "Berries", 633: "Fruit salad and cocktail",
    634: "Guacamole", 641: "Fruit juice", 642: "Fruit nectar",
    644: "Vinegar", 671: "Baby food fruit", 672: "Baby food juice",
    673: "Baby food bananas", 674: "Baby food fruit dessert", 675: "Baby food fruit and meat",
    676: "Baby food fruit and vegetables", 710: "Potato unspecified", 711: "Baked or boiled potato",
    712: "Potato chips", 713: "Scalloped potatoes", 714: "French fries and hash browns",
    715: "Mashed potatoes", 716: "Potato salad", 717: "Stewed potatoes",
    718: "Potato soup", 719: "Plantain and cassava", 721: "Leafy greens",
    722: "Broccoli", 723: "Green vegetable soup", 731: "Carrots",
    732: "Pumpkin", 733: "Winter squash", 734: "Sweet potato",
    735: "Squash soup", 741: "Raw tomatoes", 742: "Cooked tomatoes",
    743: "Tomato juice", 744: "Ketchup and salsa", 745: "Tomato salad",
    746: "Tomato soup", 747: "Tomato sandwich", 751: "Salad vegetables",
    752: "Cooked vegetables", 753: "Mixed vegetables", 754: "Onion rings and vegetable dishes",
    755: "Mustard pickles and olives", 756: "Vegetable soup", 761: "Baby food spinach",
    762: "Baby food vegetables", 764: "Baby food mixed vegetables", 766: "Baby food vegetables and meat",
    771: "Stuffed potatoes", 772: "Plantain fritters", 773: "Stuffed cabbage",
    775: "Spanish stew", 781: "Smoothies and vegetable juice", 811: "Butter and margarine",
    812: "Ghee and animal fat", 813: "Tartar sauce and spreads", 821: "Cooking oils",
    831: "Mayonnaise and dressings", 832: "Light dressings", 833: "Fat free dressings",
    899: "Sandwich toppings", 911: "Sugar", 912: "Sugar substitutes",
    913: "Syrup and honey", 914: "Jam and jelly", 915: "Gelatin dessert",
    916: "Popsicles and ices", 917: "Candy", 918: "Chewing gum",
    921: "Coffee", 923: "Tea", 924: "Soft drinks",
    925: "Fruit drinks", 926: "Horchata and grain drinks", 928: "Nonalcoholic malt drinks",
    929: "Drink mixes", 931: "Beer", 932: "Liqueur",
    933: "Cocktails", 934: "Wine", 935: "Liquor",
    940: "Tap water", 941: "Bottled water", 942: "Flavored water",
    943: "Baby water", 951: "Nutrition shakes", 952: "Protein powder",
    953: "Sports and energy drinks",
}

In [3]:
if "google.colab" in sys.modules and not os.environ.get("NHANES_DIR"):
    from google.colab import drive
    drive.mount("/content/drive")

assert os.path.isdir(DATA_DIR), f"Folder not found: {DATA_DIR}. Put the .xpt files there or edit DATA_DIR."
os.makedirs(OUT_DIR, exist_ok=True)
print(sorted(os.listdir(DATA_DIR)))

Mounted at /content/drive
['DEMO_J.xpt', 'DIQ_J.xpt', 'DR1IFF_J.xpt', 'DR2IFF_J.xpt', 'DRXFCD_J.xpt', 'GLU_J.xpt', 'food_group_item_baskets.csv', 'food_group_items.csv', 'food_group_nutrition.csv', 'glucose_food_model.json', 'glucose_food_training_data.csv', 'glucose_food_training_data_v2.csv', 'glucose_food_training_data_v3.csv', 'glucose_food_training_data_v4.csv', 'results']


### Step 2: Load the XPT files and clean what the format leaves behind

In [4]:
def read_xpt(name):
    path = os.path.join(DATA_DIR, name + ".xpt")
    df = pd.read_sas(path, format="xport", encoding="latin1")
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]):
            tiny = (df[c].abs() < 1e-70) & (df[c] != 0)
            df.loc[tiny, c] = np.nan
    for c in ("SEQN", "DRXFDCD", "DR1IFDCD", "DR2IFDCD"):
        if c in df.columns:
            df[c] = df[c].round().astype("Int64")
    for c in df.columns:
        if df[c].dtype == object or str(df[c].dtype) in ("str", "string"):
            df[c] = df[c].astype(str).str.strip()
    return df

glu  = read_xpt("GLU_J")
demo = read_xpt("DEMO_J")
diq  = read_xpt("DIQ_J")
fcd  = read_xpt("DRXFCD_J")

EXPECTED = {"GLU_J": 3036, "DEMO_J": 9254, "DIQ_J": 8897, "DRXFCD_J": 7083}
for name, df in [("GLU_J", glu), ("DEMO_J", demo), ("DIQ_J", diq), ("DRXFCD_J", fcd)]:
    flag = "OK" if len(df) == EXPECTED[name] else "<-- CHECK: expected %d" % EXPECTED[name]
    print(f"{name:9s} rows={len(df):>6,d}  cols={df.shape[1]:>3d}  {flag}")

GLU_J     rows= 3,036  cols=  4  OK
DEMO_J    rows= 9,254  cols= 46  OK
DIQ_J     rows= 8,897  cols= 54  OK
DRXFCD_J  rows= 7,083  cols=  3  OK


### Step 3: Quick checks on missing values and duplicates

In [5]:
print("Missing glucose values:", glu["LBXGLU"].isna().sum(), "of", len(glu))
print("Duplicate SEQN  GLU/DEMO/DIQ:", glu.SEQN.duplicated().sum(), demo.SEQN.duplicated().sum(), diq.SEQN.duplicated().sum())
print("Duplicate food codes in lookup:", fcd.DRXFDCD.duplicated().sum())
print()
print(glu["LBXGLU"].describe().round(1))

Missing glucose values: 145 of 3036
Duplicate SEQN  GLU/DEMO/DIQ: 0 0 0
Duplicate food codes in lookup: 0

count    2891.0
mean      111.8
std        35.5
min        47.0
25%        96.0
50%       102.0
75%       113.0
max       451.0
Name: LBXGLU, dtype: float64


## Part 2: Build the person table
### Step 4: Keep valid adults

In [6]:
funnel = {}
people = glu[["SEQN", "LBXGLU"]].dropna()
funnel["Valid fasting glucose"] = len(people)

people = people.merge(demo[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG"]], on="SEQN", how="inner")
people = people[people.RIDAGEYR >= MIN_AGE]
funnel[f"Age >= {MIN_AGE}"] = len(people)

people = people[~people.RIDEXPRG.eq(1)]
funnel["Not pregnant"] = len(people)

people = people.merge(diq[["SEQN", "DIQ010", "DIQ050", "DIQ070"]], on="SEQN", how="left")
people["diagnosed"] = (people.DIQ010.eq(1) | people.DIQ050.eq(1) | people.DIQ070.eq(1))
people["SEQN"] = people.SEQN.astype(int)
people = people.reset_index(drop=True)
print(pd.Series(funnel).to_frame("people"))
print("\nDiagnosed diabetes / insulin / pills:", int(people.diagnosed.sum()))

                       people
Valid fasting glucose    2891
Age >= 18                2537
Not pregnant             2517

Diagnosed diabetes / insulin / pills: 450


### Step 5: Load the food records and keep reliable recalls only

In [7]:
NUTS = ["grams", "kcal", "carb", "sugar", "fiber", "satfat"]

def load_food_day(day):
    d = read_xpt(f"DR{day}IFF_J")
    expected = {1: 112683, 2: 93500}[day]
    print(f"DR{day}IFF_J rows={len(d):,d}  {'OK' if len(d)==expected else '<-- CHECK expected %d' % expected}")
    keep = {"SEQN": "SEQN", f"DR{day}DRSTZ": "status", f"DR{day}IFDCD": "code",
            f"DR{day}IGRMS": "grams", f"DR{day}IKCAL": "kcal", f"DR{day}ICARB": "carb",
            f"DR{day}ISUGR": "sugar", f"DR{day}IFIBE": "fiber", f"DR{day}ISFAT": "satfat"}
    d = d[list(keep)].rename(columns=keep)
    d["day"] = day
    return d

food = pd.concat([load_food_day(d) for d in DAYS_TO_USE], ignore_index=True)
food = food[food.status.eq(1)].dropna(subset=["code", "grams"]).copy()
food["SEQN"] = food.SEQN.astype(int)
food["code"] = food.code.astype(int)
food[NUTS] = food[NUTS].fillna(0)

names = fcd.assign(code=fcd.DRXFDCD.astype(int)).set_index("code")["DRXFCSD"]
food["name"] = food.code.map(names)
print("\nFood rows kept:", f"{len(food):,d}", "| rows without a name:", int(food.name.isna().sum()))

food["group"] = food.code // 100000
print("Distinct foods:", food.code.nunique(), "-> distinct 3-digit groups:", food.group.nunique())

DR1IFF_J rows=112,683  OK
DR2IFF_J rows=93,500  OK

Food rows kept: 202,670 | rows without a name: 0
Distinct foods: 5126 -> distinct 3-digit groups: 220


## Part 3: Groups and baskets
### Step 6: Person features and glucose bands

In [8]:
people["band"] = pd.cut(people.LBXGLU, bins=BAND_CUTS, labels=BAND_LABELS, right=False).astype(str)
print(pd.crosstab(people.band, people.diagnosed, margins=True).reindex(BAND_LABELS + ["All"]))

people["Age"] = people.RIDAGEYR.astype(int)
people["Gender"] = people.RIAGENDR.map({1: "Male", 2: "Female"})
print("\nMissing values in the person features:")
print(people[["Age", "Gender"]].isna().sum().to_string())

diagnosed                    False  True   All
band                                          
Normal (<100)                  871    33   904
Prediabetic range (100-125)   1087   118  1205
Diabetic range (>=126)         109   299   408
All                           2067   450  2517

Missing values in the person features:
Age       0
Gender    0


### Step 7: Group the foods

In [9]:
def clean_name(s):
    return s.capitalize() if isinstance(s, str) else str(s)

by_group_names = food.groupby("group")["name"].agg(lambda s: [clean_name(x) for x in s.value_counts().index])
group_examples = {g: names_[:3] for g, names_ in by_group_names.items()}
group_label = {g: " / ".join(n[:2])[:60] for g, n in by_group_names.items()}
group_label.update(GROUP_NAME_OVERRIDES)

def col_name(g):
    safe = re.sub(r"[^A-Za-z0-9]+", "_", group_label.get(g, str(g))).strip("_")
    return f"{safe}_{g}"

sums = food.groupby("group")[NUTS].sum()
nut = pd.DataFrame(index=sums.index)
for c in ["kcal", "carb", "sugar", "fiber", "satfat"]:
    nut[c + "100"] = sums[c] / sums["grams"] * 100
nut["netcarb100"] = nut.carb100 - nut.fiber100
nut = nut.replace([np.inf, -np.inf], np.nan)
print(nut.loc[[g for g in (631, 511, 924, 311, 421) if g in nut.index]].round(1)
        .assign(column=[col_name(g) for g in (631, 511, 924, 311, 421) if g in nut.index]))

       kcal100  carb100  sugar100  fiber100  satfat100  netcarb100                     column
group                                                                                        
631       62.9     15.4      10.6       2.0        0.1        13.4            Fresh_fruit_631
511      303.7     49.8       7.8       2.2        2.4        47.6  White_bread_and_rolls_511
924       33.2      8.3       7.9       0.0        0.0         8.3            Soft_drinks_924
311      165.9      0.7       0.4       0.0        3.7         0.7   Eggs_boiled_or_fried_311
421      598.4     22.1       5.6       8.0        7.1        14.1                   Nuts_421


### Step 8: Remove noise items and build the baskets

In [10]:
food_nn = food[~food.group.isin(NOISE_GROUPS)]
person_items = food_nn.groupby("SEQN")["group"].agg(lambda s: frozenset(s.tolist())).to_dict()

people = people[people.SEQN.isin(person_items)].reset_index(drop=True)
print("People with glucose + at least one counted food:", len(people))
sizes = pd.Series({s: len(person_items[s]) for s in people.SEQN})
print("Basket size (distinct food groups per person):"); print(sizes.describe().round(1).to_string())

main_pop = people[~people.diagnosed].reset_index(drop=True) if EXCLUDE_DIAGNOSED else people.copy()
print("\nPeople in the final dataset:", len(main_pop))
print(main_pop.band.value_counts().reindex(BAND_LABELS))

People with glucose + at least one counted food: 2293
Basket size (distinct food groups per person):
count    2293.0
mean       14.5
std         5.3
min         1.0
25%        11.0
50%        14.0
75%        18.0
max        34.0

People in the final dataset: 1885
band
Normal (<100)                  797
Prediabetic range (100-125)    994
Diabetic range (>=126)          94
Name: count, dtype: int64


### Step 9: Nutrition filter

In [11]:
def group_suitable(g, band):
    if g not in nut.index or nut.loc[g].isna().any(): return False
    if g in ALWAYS_EXCLUDE or g in PROCESSED_EXCLUDE: return False
    if g // 10 == ALCOHOL_PREFIX or g // 10 in FATS_PREFIXES: return False
    if band != BAND_LABELS[0] and g in HIGHER_BAND_EXCLUDE: return False
    n = nut.loc[g]; lim = NUTRI_LIMITS[band]
    sugar_ok = (n.sugar100 <= lim["sugar"]) or (n.fiber100 >= FIBER_EXCEPTION["min_fiber"] and n.sugar100 <= FIBER_EXCEPTION["max_sugar"])
    return bool(sugar_ok and n.netcarb100 <= lim["netcarb"] and n.satfat100 <= SATFAT_MAX)

used_groups = sorted({g for s in main_pop.SEQN for g in person_items[s]})
SUITABLE = {b: [g for g in used_groups if group_suitable(g, b)] for b in BAND_LABELS}
print("Food groups in the dataset:", len(used_groups))
print(pd.DataFrame({b: [len(SUITABLE[b])] for b in BAND_LABELS}, index=["suitable groups"]))

print("\n# ---- copy these into your training notebook (group codes suitable for each band) ----")
for b, short in zip(BAND_LABELS, ["NORMAL", "PREDIABETIC", "DIABETIC"]):
    print(f"SUITABLE_{short} = {SUITABLE[b]}")

Food groups in the dataset: 184
                 Normal (<100)  Prediabetic range (100-125)  Diabetic range (>=126)
suitable groups            122                           98                      94

# ---- copy these into your training notebook (group codes suitable for each band) ----
SUITABLE_NORMAL = [111, 113, 114, 115, 121, 123, 134, 142, 145, 146, 147, 200, 210, 211, 214, 215, 220, 221, 222, 223, 224, 227, 230, 231, 232, 233, 242, 243, 244, 251, 252, 261, 262, 271, 272, 273, 274, 281, 283, 285, 311, 312, 321, 322, 323, 324, 330, 334, 411, 412, 413, 414, 416, 418, 421, 422, 423, 424, 431, 510, 511, 513, 514, 515, 516, 518, 522, 540, 551, 552, 553, 555, 556, 557, 561, 562, 576, 583, 584, 590, 611, 612, 631, 632, 633, 634, 641, 642, 644, 710, 711, 713, 715, 716, 717, 718, 719, 721, 722, 723, 731, 732, 733, 734, 735, 741, 742, 743, 745, 746, 751, 752, 753, 756, 772, 775, 781, 915, 926, 928, 951, 952]
SUITABLE_PREDIABETIC = [111, 113, 121, 123, 134, 142, 145, 146, 147, 200, 210, 211

## Part 4: The single dataset
### Step 10: Build the one table

In [12]:
long = (food_nn[food_nn.SEQN.isin(main_pop.SEQN)][["SEQN", "group"]].drop_duplicates())
onehot = pd.crosstab(long.SEQN, long.group).clip(upper=1).astype(int)
onehot = onehot[sorted(onehot.columns)]
onehot.columns = [col_name(g) for g in onehot.columns]

base = main_pop[["SEQN", "LBXGLU", "Age", "Gender"]].rename(columns={"LBXGLU": "Glucose_mg_dL"})

dataset = base.merge(onehot, left_on="SEQN", right_index=True, how="inner").sort_values("SEQN").reset_index(drop=True)
FOOD_COLS = list(onehot.columns)

print("Dataset shape:", dataset.shape, f"({len(FOOD_COLS)} food-group columns)")
print("Duplicate SEQN:", int(dataset.SEQN.duplicated().sum()))
print("Missing values:", int(dataset.isna().sum().sum()))
dataset.iloc[:, :8].head()

Dataset shape: (1885, 188) (184 food-group columns)
Duplicate SEQN: 0
Missing values: 0


,SEQN,Glucose_mg_dL,Age,Gender,Milk_111,Evaporated_or_condensed_milk_112,Almond_or_soy_milk_113,Yogurt_114
0,93708,122.0,66,Female,1,0,0,0
1,93711,107.0,56,Male,1,0,0,1
2,93717,91.0,22,Male,0,0,0,0
3,93721,104.0,60,Female,1,0,0,0
4,93722,101.0,60,Female,1,0,0,0


### Step 11: Save and download the CSVs

In [13]:
out_path = os.path.join(OUT_DIR, OUT_FILE)
dataset.to_csv(out_path, index=False)
print("Saved:", out_path, "|", dataset.shape[0], "rows x", dataset.shape[1], "columns")

codes = [int(c.rsplit("_", 1)[1]) for c in FOOD_COLS]
nutrition = pd.DataFrame({"code": codes, "column": FOOD_COLS, "name": [group_label.get(c, str(c)) for c in codes],
                          "kcal_100g": nut.loc[codes, "kcal100"].round(2).values,
                          "carb_100g": nut.loc[codes, "carb100"].round(2).values,
                          "sugar_100g": nut.loc[codes, "sugar100"].round(2).values,
                          "fiber_100g": nut.loc[codes, "fiber100"].round(2).values,
                          "net_carb_100g": nut.loc[codes, "netcarb100"].round(2).values,
                          "sat_fat_100g": nut.loc[codes, "satfat100"].round(2).values})
nut_path = os.path.join(OUT_DIR, NUT_FILE)
nutrition.to_csv(nut_path, index=False)
print("Saved:", nut_path, "|", len(nutrition), "food groups")
print(nutrition.head().to_string(index=False))

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(out_path)
    files.download(nut_path)

Saved: /content/drive/MyDrive/NHANES/glucose_food_training_data_v3.csv | 1885 rows x 188 columns
Saved: /content/drive/MyDrive/NHANES/food_group_nutrition.csv | 184 food groups
 code                           column                         name  kcal_100g  carb_100g  sugar_100g  fiber_100g  net_carb_100g  sat_fat_100g
  111                         Milk_111                         Milk      52.46       4.85        4.88        0.00           4.85          1.30
  112 Evaporated_or_condensed_milk_112 Evaporated or condensed milk     173.04      19.91       19.91        0.00          19.91          4.62
  113           Almond_or_soy_milk_113           Almond or soy milk      28.58       3.77        2.94        0.20           3.57          0.20
  114                       Yogurt_114                       Yogurt      85.70      12.11       10.73        0.21          11.90          1.04
  115               Chocolate_milk_115               Chocolate milk      81.23      12.38       10.06       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Step 11b: The specific foods inside each group

In [14]:
pop_food = food_nn[food_nn.SEQN.isin(dataset.SEQN) & food_nn.group.isin(codes)]

per_item = pop_food.groupby(["group", "code"])[NUTS].sum()
per_item["people"] = pop_food.groupby(["group", "code"])["SEQN"].nunique()
per_item = per_item.reset_index()
per_item = per_item[(per_item.grams > 0) & (per_item.people >= ITEM_MIN_PEOPLE)].copy()

for c in ["kcal", "carb", "sugar", "fiber", "satfat"]:
    per_item[c + "100"] = per_item[c] / per_item.grams * 100
per_item["netcarb100"] = per_item.carb100 - per_item.fiber100

per_item = (per_item.sort_values(["group", "people", "code"], ascending=[True, False, True])
                    .groupby("group").head(ITEM_TOP_N))

long_names = fcd.assign(code=fcd.DRXFDCD.astype(int)).set_index("code")["DRXFCLD"]
group_items = pd.DataFrame({
    "group": per_item.group.astype(int).values,
    "code": per_item.code.astype(int).values,
    "name": per_item.code.map(long_names).fillna(per_item.code.map(names).map(clean_name)).values,
    "people": per_item.people.astype(int).values,
    "kcal_100g": per_item.kcal100.round(2).values,
    "carb_100g": per_item.carb100.round(2).values,
    "sugar_100g": per_item.sugar100.round(2).values,
    "fiber_100g": per_item.fiber100.round(2).values,
    "net_carb_100g": per_item.netcarb100.round(2).values,
    "sat_fat_100g": per_item.satfat100.round(2).values,
})

items_path = os.path.join(OUT_DIR, ITEMS_FILE)
group_items.to_csv(items_path, index=False)
print("Saved:", items_path, "|", len(group_items), "specific foods in", group_items.group.nunique(), "groups")

item_baskets = (pop_food[pop_food.code.isin(group_items.code)][["SEQN", "code"]]
                .drop_duplicates().sort_values(["SEQN", "code"]).reset_index(drop=True))
baskets_path = os.path.join(OUT_DIR, ITEM_BASKETS_FILE)
item_baskets.to_csv(baskets_path, index=False)
print("Saved:", baskets_path, "|", len(item_baskets), "rows,", item_baskets.SEQN.nunique(), "people")
print(group_items[group_items.group == 631].head(10).to_string(index=False))

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(items_path)
    files.download(baskets_path)

Saved: /content/drive/MyDrive/NHANES/food_group_items.csv | 964 specific foods in 156 groups
Saved: /content/drive/MyDrive/NHANES/food_group_item_baskets.csv | 25391 rows, 1882 people
 group     code            name  people  kcal_100g  carb_100g  sugar_100g  fiber_100g  net_carb_100g  sat_fat_100g
   631 63107010     Banana, raw     406      88.89      22.84       12.23        2.61          20.23          0.11
   631 63101000      Apple, raw     225      52.03      13.81       10.39        2.40          11.41          0.03
   631 63123000     Grapes, raw     164      69.05      18.10       15.48        0.91          17.19          0.05
   631 63105010    Avocado, raw     111     160.00       8.53        0.66        6.70           1.83          2.13
   631 63149010 Watermelon, raw      75      30.00       7.55        6.20        0.40           7.15          0.02
   631 63109010 Cantaloupe, raw      68      34.06       8.16        7.88        0.80           7.36          0.05
   631 6314

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Part 5: Check that the dataset can recommend foods per glucose band

In [15]:
df = pd.read_csv(out_path)
META = ["SEQN", "Glucose_mg_dL", "Age", "Gender"]
food_cols = [c for c in df.columns if c not in META]

band = pd.cut(df.Glucose_mg_dL, bins=BAND_CUTS, labels=BAND_LABELS, right=False).astype(str)
print("Transactions (people) per glucose band:")
print(band.value_counts().reindex(BAND_LABELS).to_string())

sub = df[band == BAND_LABELS[0]]
rows = [[c for c in food_cols if r[c] == 1] for _, r in sub.iterrows()]
pyeclat_df = pd.DataFrame(rows)
print("pyECLAT input        :", pyeclat_df.shape)

mlxtend_df = df[food_cols].astype(bool)
print("mlxtend input        :", mlxtend_df.shape)

X, y = df[food_cols + ["Age", "Gender"]], band
print("scikit-learn X, y    :", X.shape, y.shape)
pyeclat_df.head()

Transactions (people) per glucose band:
Glucose_mg_dL
Normal (<100)                  797
Prediabetic range (100-125)    994
Diabetic range (>=126)          94
pyECLAT input        : (797, 33)
mlxtend input        : (1885, 184)
scikit-learn X, y    : (1885, 186) (1885,)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32
0,Cheese_dip_and_grilled_cheese_146,Chicken_and_chicken_nuggets_241,Deli_meats_and_sausages_252,Meat_and_poultry_in_sauce_271,Meat_loaf_and_meat_with_rice_272,White_bread_and_rolls_511,Cookies_and_brownies_532,Pizza_and_pasta_dishes_581,Fresh_fruit_631,Potato_chips_712,Raw_tomatoes_741,Salad_vegetables_751,Mayonnaise_and_dressings_831,Fat_free_dressings_833,Candy_917,Soft_drinks_924,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,Cream_121,Ice_cream_131,White_sauce_or_gravy_134,Chicken_and_chicken_nuggets_241,Shrimp_263,Burgers_and_sandwiches_275,Biscuits_521,Pies_533,Corn_chips_and_pretzels_544,Fruit_juice_641,French_fries_and_hash_browns_714,Candy_917,Fruit_drinks_925,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,Milk_111,Beef_steak_211,Chicken_and_chicken_nuggets_241,Meat_and_vegetable_soups_283,Beans_411,Peanut_butter_sandwich_423,White_bread_and_rolls_511,Biscuits_521,Tortillas_and_cornbread_522,Rice_and_oatmeal_562,Pizza_and_pasta_dishes_581,Fresh_fruit_631,French_fries_and_hash_browns_714,Salad_vegetables_751,Smoothies_and_vegetable_juice_781,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
3,Meat_and_poultry_in_sauce_271,Burgers_and_sandwiches_275,Mashed_potatoes_715,Onion_rings_and_vegetable_dishes_754,Mayonnaise_and_dressings_831,Soft_drinks_924,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
4,Natural_cheese_141,Roast_beef_214,Fish_261,Meat_and_poultry_in_sauce_271,Burgers_and_sandwiches_275,Meatless_burgers_and_bacon_bits_418,Nuts_421,White_bread_and_rolls_511,Pastries_and_fritters_534,Rice_and_oatmeal_562,Pizza_and_pasta_dishes_581,Citrus_fruit_611,Citrus_juice_612,Broccoli_722,Cooked_vegetables_752,Mixed_vegetables_753,Butter_and_margarine_811,Candy_917,Tea_923,None,None,None,None,None,None,None,None,None,None,None,None,None,None
